# 1 - Instalação e Carregamento dos Dados de Treino/Validação

In [1]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl

import os
import json
import torch
from google.colab import userdata
from huggingface_hub import login
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Login Hugging Face
login(token=userdata.get('HF_TOKEN'))

# Carregar splits de treino (4.000) e validação (500)
with open("data/train_set_4000.json", "r", encoding="utf-8") as f:
    train_raw = json.load(f)

with open("data/val_set_500.json", "r", encoding="utf-8") as f:
    val_raw = json.load(f)

with open("data/valid_intents.json", "r", encoding="utf-8") as f:
    valid_intents = json.load(f)

INTENTS_STR = ", ".join(valid_intents)
print(f"Treino: {len(train_raw)} | Validação: {len(val_raw)} | Intenções: {len(valid_intents)}")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
Treino: 4000 | Validação: 500 | Intenções: 27


# 2 - Formatação em ChatML (Prompt de Instrução + Alvo JSON)

In [2]:
SYSTEM_PROMPT = f"""You are a customer support AI assistant.
Your task is to:
1. Classify the customer's message into EXACTLY ONE of the following 27 valid intents:
[{INTENTS_STR}]
2. Write a helpful, polite, and concise customer support response.

Respond ONLY with a valid JSON object using this exact format:
{{"intent": "<exact_intent_name>", "response": "<your_reply>"}}"""

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_to_chatml(example):
    target_json = json.dumps({
        "intent": example["intent"],
        "response": example["response"]
    }, ensure_ascii=False)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": target_json}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_dataset = Dataset.from_list(train_raw).map(format_to_chatml)
val_dataset = Dataset.from_list(val_raw).map(format_to_chatml)

print("Exemplo formatado:\n", train_dataset[0]["text"][:500], "...")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Exemplo formatado:
 <|im_start|>system
You are a customer support AI assistant.
Your task is to:
1. Classify the customer's message into EXACTLY ONE of the following 27 valid intents:
[cancel_order, change_order, change_shipping_address, check_cancellation_fee, check_invoice, check_payment_methods, check_refund_policy, complaint, contact_customer_service, contact_human_agent, create_account, delete_account, delivery_options, delivery_period, edit_account, get_invoice, get_refund, newsletter_subscription, payment_is ...


# 3 - Configuração do Modelo Base (4-bit QLoRA) e Adaptadores LoRA

In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(base_model, lora_config)

# Garante que nenhum parâmetro treinável fique em BFloat16 na GPU T4
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float16)

model.print_trainable_parameters()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


# 4 - Treinamento Supervisionado (SFT) e Salvamento dos Adaptadores

In [9]:
OUTPUT_DIR = "lora_adapters/qwen2.5-1.5b-support"
os.makedirs("results", exist_ok=True)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=4,
    eval_strategy="steps",
    eval_steps=125,
    save_strategy="steps",
    save_steps=125,
    logging_steps=25,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=25,
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    max_length=512,
    dataset_text_field="text",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    args=sft_config
)

train_result = trainer.train()

# Salvar o melhor adaptador LoRA final e o tokenizador
FINAL_ADAPTER_PATH = "lora_adapters/final_model"
trainer.model.save_pretrained(FINAL_ADAPTER_PATH)
tokenizer.save_pretrained(FINAL_ADAPTER_PATH)

# Salvar log de histórico de treino para gerar o gráfico da curva de Loss depois
with open("results/training_history.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=2)

print(f"\nTreinamento concluído! Adaptadores salvos em: {FINAL_ADAPTER_PATH}")

Adding EOS to train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

... (2 outputs left out)

Truncating eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
125,0.370388,0.361181,0.352091,0.893456,664583.000000
250,0.321261,0.322240,0.325868,0.902064,1331816.000000
375,0.285905,0.308842,0.294949,0.905188,1995438.000000
500,0.290993,0.305938,0.295633,0.906012,2663632.000000



Treinamento concluído! Adaptadores salvos em: lora_adapters/final_model


In [10]:
!zip -r final_model.zip lora_adapters/final_model

  adding: lora_adapters/final_model/ (stored 0%)
  adding: lora_adapters/final_model/tokenizer.json (deflated 81%)
  adding: lora_adapters/final_model/chat_template.jinja (deflated 71%)
  adding: lora_adapters/final_model/adapter_config.json (deflated 61%)
  adding: lora_adapters/final_model/adapter_model.safetensors (deflated 53%)
  adding: lora_adapters/final_model/README.md (deflated 65%)
  adding: lora_adapters/final_model/tokenizer_config.json (deflated 59%)
